In [0]:
from pyspark.sql import functions as F

# 1. CONFIGURATION
SILVER_PO_TABLE = "mini_project_team_a3t7.silver.purchase_order"
GOLD_SUPPLIER_KPI_TABLE = "mini_project_team_a3t7.gold.supplier_performance_fact"
CHECKPOINT_SUPPLIER_KPI = "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/gold/supplier_kpis_stream"

# 2. READ STREAM FROM SILVER
df_po_stream = spark.readStream.table(SILVER_PO_TABLE)

# 3. ENRICH DATA (Fixing the Missing Column Error)
# We calculate the delay days before the aggregation happens
df_enriched = (
    df_po_stream
    .withColumn("delivery_delay_days", 
        F.datediff(F.col("actual_delivery_date"), F.col("expected_delivery_date")))
)

# 4. KPI AGGREGATIONS
supplier_kpis = (
    df_enriched
    .groupBy("supplier_id")
    .agg(
        # KPI 1: On-time Delivery %
        (F.sum(F.when(F.col("delivery_delay_days") <= 0, 1).otherwise(0)) / 
         F.count("*") * 100).alias("otd_pct"),

        # KPI 2: Lead Time Variance
        F.round(F.stddev("delivery_delay_days"), 2).alias("lead_time_variance"),

        # KPI 3: Average & Quality Metrics
        F.round(F.avg("delivery_delay_days"), 2).alias("avg_delay_days"),
        F.round(F.avg("quality_rating"), 2).alias("avg_quality_rating"),

        # Volume Metrics
        F.count("*").alias("total_orders"),
        F.round(F.sum(F.col("quantity_ordered") * F.col("unit_cost")), 2).alias("total_po_value")
    )
    
    # --- Composite Supplier Risk Score Logic ---
    .withColumn("norm_otd", F.col("otd_pct") / 100)
    .withColumn("norm_quality", F.coalesce(F.col("avg_quality_rating"), F.lit(0.0)) / 5.0)
    .withColumn("norm_lt_var", 
        F.when(F.col("lead_time_variance") > 0, F.lit(1.0) - (F.col("lead_time_variance") / 10.0))
         .otherwise(F.lit(1.0)))
    
    .withColumn("supplier_risk_score", 
        F.round(F.col("norm_otd") * 0.4 + F.col("norm_quality") * 0.3 + F.col("norm_lt_var") * 0.3, 3))
    
    .withColumn("actual_risk_tier",
        F.when(F.col("supplier_risk_score") > 0.55, "Low")
         .when(F.col("supplier_risk_score") > 0.53, "Medium")
         .otherwise("High"))
    
    .drop("norm_otd", "norm_quality", "norm_lt_var")
    .withColumn("_gold_processed_at", F.current_timestamp())
)

# 5. WRITE STREAM & DISPLAY
# 'complete' mode ensures the table always contains the most recent global summary
query = (supplier_kpis.writeStream
    .format("delta")
    .outputMode("complete") 
    .option("checkpointLocation", CHECKPOINT_SUPPLIER_KPI)
    .trigger(availableNow=True)
    .toTable(GOLD_SUPPLIER_KPI_TABLE))

query.awaitTermination()

# 6. DISPLAY RESULTS
# print(f"Supplier Performance Table: {GOLD_SUPPLIER_KPI_TABLE}")
# display(spark.table(GOLD_SUPPLIER_KPI_TABLE).orderBy(F.col("supplier_risk_score").desc()))

In [0]:
# supplier_kpis.display()